# Poisson-GLM

This notebook implements Generalized Linear Model (GLM) analysis inspired by the neuroGLM toolkit (https://github.com/memming/neuroGLM) and the methodology from:

**Park et al. (2014). "Encoding and decoding in parietal cortex during sensorimotor decision-making." Nature Neuroscience 17, 1395-1403.**



In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import time
from scipy.io import loadmat
from scipy import stats, signal
from scipy.sparse import csr_matrix, vstack, hstack, issparse
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')


# Set style for publication-quality plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")


In [ ]:
from imports import *
from src.utils import poisson_glm_utils
from config import dir_config


In [ ]:
compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)
result_dir = Path(processed_dir, 'poisson_glm', 'equal_block_cross_validation')

session_to_exclude = ["210210_GP_JP", "241209_GP_TZ"]

session_metadata = pd.read_csv(Path(processed_dir, 'sessions_metadata.csv'))
session_metadata = session_metadata[~np.isin(session_metadata["session_id"], session_to_exclude)]

neuron_metadata = pd.read_csv(Path(processed_dir, 'neuron_metadata.csv'))
neuron_metadata = neuron_metadata[~np.isin(neuron_metadata["session_id"], session_to_exclude)].reset_index()

with open(Path(processed_dir, f'glm_hmm_models', f'glm_hmm_masked_final.pkl'), 'rb') as f:
    glm_hmm = pickle.load(f)

with open(Path(result_dir, 'config.pkl'), 'rb') as f:
    poisson_glm_config = pickle.load(f)

In [ ]:
# Feature indices for easy access
feature_idx = {
    'target_start': 0,
    'target_end': poisson_glm_config.FEATURES_TARGET,
    'stim_start': poisson_glm_config.FEATURES_TARGET,
    'stim_end': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS,
    'stim_offset_start': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS,
    'stim_offset_end': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS + poisson_glm_config.FEATURE_STIMULUS_OFFSET,
    'saccade_start': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS + poisson_glm_config.FEATURE_STIMULUS_OFFSET,
    'saccade_end': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS + poisson_glm_config.FEATURE_STIMULUS_OFFSET + poisson_glm_config.FEATURES_SACCADE,
    'history_start': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS + poisson_glm_config.FEATURE_STIMULUS_OFFSET + poisson_glm_config.FEATURES_SACCADE,
    'history_end': poisson_glm_config.FEATURES_TARGET + poisson_glm_config.FEATURES_STIMULUS + poisson_glm_config.FEATURE_STIMULUS_OFFSET + poisson_glm_config.FEATURES_SACCADE + poisson_glm_config.FEATURES_HISTORY,
    'intercept_idx': poisson_glm_config.get_total_features() - 1
}

print(f"Total features: {poisson_glm_config.get_total_features()}")

In [ ]:
coh_levels=np.array([-0.5 , -0.2 , -0.06,  0.  ,  0.06,  0.2 ,  0.5 ])

In [ ]:
bad_neurons = ['31',
 '13',
 '140',
 '141',
 '19',
 '20',
 '44',
 '76',
 '24',
 '48',
 '112',
 '104',
 '88',
 '15',
 '138',
 '46',
 '81',
 '49',
 '166',
 '139',
 '55',
 '38',
 '90',
 '6',
 '10',
 '52',
 '152',
 '63']

### Helper Functions

In [ ]:
def extract_neuron_data(neuron_id):
    session_name = neuron_metadata.loc[neuron_metadata["neuron_id"] == neuron_id, "session_id"].values[0]
    data_path = Path(compiled_dir, session_name)

    print(f"Analyzing neuron {neuron_id} from session {session_name}")

    # Load neural and behavioral data
    try:
        spike_times = np.load(data_path / "spike_times.npy")
        spike_clusters = np.load(data_path / "spike_clusters.npy")
    except:
        spike_times = loadmat(Path(compiled_dir, session_name, "spike_times.mat"))
        spike_times = spike_times["spike_times"][0]
        spike_clusters = loadmat(Path(compiled_dir, session_name, "spike_clusters.mat"))
        spike_clusters = spike_clusters["spike_clusters"][0]

    # Get neuron spike times
    cluster_id = neuron_metadata.cluster[neuron_metadata["neuron_id"] == neuron_id].values[0]
    neuron_spike_times = spike_times[spike_clusters == cluster_id]
    neuron_spike_times = (neuron_spike_times / 30).round().astype(int)  # Convert to ms

    # Get timestamps and trial data
    timestamps = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_timestamps.csv"), index_col=None)
    trial_info = pd.read_csv(Path(compiled_dir, session_name, f"{session_name}_trial.csv"), index_col=None)

    # Process trial data
    GP_trial_data = trial_info[trial_info.task_type == 1].reset_index(drop=True)
    # signed coherence
    GP_trial_data["signed_coherence"] = GP_trial_data["coherence"] * (2*GP_trial_data["target"]-1)
    
    GP_trial_data = GP_trial_data[GP_trial_data.reaction_time.notna()]
    # include equal block only
    GP_trial_data = GP_trial_data[GP_trial_data.prob_toRF == 50]
    GP_trial_data["state"] = 0

    coh_levels = np.sort(GP_trial_data['signed_coherence'].unique()) /100  # Normalize coherence

    # Keep only correct trials
    GP_trial_data = GP_trial_data[GP_trial_data.outcome == 1].reset_index()

    # print(f"Found {len(GP_trial_data)} valid trials")
    # print(f"Neuron has {len(neuron_spike_times)} spikes")
    # print(f"State distribution: {GP_trial_data.state.value_counts().to_dict()}")

    return GP_trial_data, neuron_spike_times, timestamps, coh_levels

def create_neuroglm_trials(session_data, timestamps, neuron_spike_times, bin_size=1.0):
    """
    Create trial structure following neuroGLM format.
    """
    trials = []

    # Convert timestamps to ms
    timestamps_ms = (timestamps / 30).round()

    for idx, row in session_data.iterrows():
        trial_idx = row.trial_number - 1  # Convert to 0-based

        # Trial timing (relative to target onset - 50ms)
        target_onset = timestamps_ms.loc[trial_idx, "target_onset"]
        trial_start = target_onset - 50
        trial_end = timestamps_ms.loc[trial_idx, "response_onset"]
        duration = trial_end - trial_start

        if pd.isna(duration) or duration <= 0:
            continue

        # Get trial spike times (relative to trial start)
        trial_spikes = neuron_spike_times[
            (neuron_spike_times >= trial_start) &
            (neuron_spike_times <= trial_end)
        ] - trial_start

        # Create binned spike train
        n_bins = int(np.ceil(duration / bin_size))
        spike_train = np.zeros(n_bins)

        for spike_time in trial_spikes:
            bin_idx = int(np.floor(spike_time / bin_size))
            if 0 <= bin_idx < n_bins:
                spike_train[bin_idx] += 1

        # Event timings (relative to trial start)
        events = {
            'target_onset': 50,  # Always 50ms into trial
            'stimulus_onset': timestamps_ms.loc[trial_idx, "stimulus_onset"] - trial_start,
            'stimulus_offset': timestamps_ms.loc[trial_idx, "response_onset"] - trial_start,
            'response_onset': timestamps_ms.loc[trial_idx, "response_onset"] - trial_start,
        }

        # Trial structure
        trial = {
            'duration': duration,
            'spike_train': spike_train,
            'n_bins': n_bins,

            # Event timings
            'target_onset': events['target_onset'],
            'stimulus_onset': events['stimulus_onset'],
            'stimulus_offset': events['stimulus_offset'],
            'response_onset': events['response_onset'],

            # Experimental variables
            'coherence': row.signed_coherence / 100,  # Normalize coherence
            'choice': row.choice,
            'state': row.state,  # Bias state from GLM-HMM
            'reaction_time': row.reaction_time,

            # Trial metadata
            'trial_idx': int(trial_idx),
            # 'original_idx': idx
        }

        trials.append(trial)

    return trials

def build_design_matrix(trials, coh_levels):
    # Initialize containers
    trial_matrices = []
    trial_spike_trains = []

    # Process each trial
    for trial in trials:
        trial_duration = int(trial['duration'])
        trial_design = np.zeros((trial_duration, poisson_glm_config.get_total_features()))

        # 1. TARGET ONSET COMPONENT
        target_bin = int(trial['target_onset'])
        if 0 < target_bin <= trial_duration:
            target_matrix = np.zeros((trial_duration, 1))
            target_matrix[target_bin-1] = 1.0
            target_conv, _ = poisson_glm_utils.convolve_with_basis(
                target_matrix,
                poisson_glm_config.TARGET_BASIS,
                poisson_glm_config.TARGET_DURATION_MS,
                poisson_glm_config.TARGET_SPACING_MS,
                effect= poisson_glm_config.TARGET_EFFECT
            )

            target_start = feature_idx['target_start'] + int(trial['state']) * poisson_glm_config.TARGET_N_BASES
            target_end = target_start + poisson_glm_config.TARGET_N_BASES
            trial_design[:, target_start:target_end] = target_conv


        # 2. STIMULUS COHERENCE COMPONENT
        stim_bin = int(trial['stimulus_onset'])
        resp_bin = int(trial['response_onset'])
        if 0 < stim_bin < resp_bin <= trial_duration:
            stim_matrix = np.zeros((trial_duration, 1))
            stim_matrix[stim_bin-1:resp_bin] = 1.0
            stim_conv, _ = poisson_glm_utils.convolve_with_basis(
                stim_matrix,
                poisson_glm_config.STIMULUS_BASIS,
                poisson_glm_config.STIMULUS_DURATION_MS,
                poisson_glm_config.STIMULUS_SPACING_MS,
                effect= poisson_glm_config.STIMULUS_EFFECT
            )

            # Assign to coherence-specific and state-specific features
            coh_idx = np.where(coh_levels == trial['coherence'])[0][0]
            state_idx = int(trial['state'])
            coh_start = feature_idx['stim_start'] + coh_idx * poisson_glm_config.STIMULUS_N_BASES + state_idx * poisson_glm_config.STIMULUS_N_BASES * len(coh_levels)
            coh_end = coh_start + poisson_glm_config.STIMULUS_N_BASES
            trial_design[:, coh_start:coh_end] = stim_conv

        # 3. STIMULUS OFFSET COHERENCE COMPONENT
        stim_offset_bin = int(trial['stimulus_offset'])
        if 0 < stim_offset_bin <= trial_duration:
            stim_offset_matrix = np.zeros((trial_duration, 1))
            stim_offset_matrix[stim_offset_bin-1] = 1.0
            stim_offset_conv, _ = poisson_glm_utils.convolve_with_basis(
                stim_offset_matrix,
                poisson_glm_config.STIMULUS_OFFSET_BASIS,
                poisson_glm_config.STIMULUS_OFFSET_DURATION_MS,
                poisson_glm_config.STIMULUS_OFFSET_SPACING_MS,
                effect= poisson_glm_config.STIMULUS_OFFSET_EFFECT
            )

            # Assign to coherence-specific and state-specific features
            coh_idx = np.where(coh_levels == trial['coherence'])[0][0]
            state_idx = int(trial['state'])
            coh_start = feature_idx['stim_offset_start'] + coh_idx * poisson_glm_config.STIMULUS_OFFSET_N_BASES + state_idx * poisson_glm_config.STIMULUS_OFFSET_N_BASES * len(coh_levels)
            coh_end = coh_start + poisson_glm_config.STIMULUS_OFFSET_N_BASES
            trial_design[:, coh_start:coh_end] = stim_offset_conv

        # 4. SACCADE/CHOICE COMPONENT
        if 0 < resp_bin <= trial_duration:
            saccade_matrix = np.zeros((trial_duration, 1))
            saccade_matrix[resp_bin-1] = 1.0
            saccade_conv, _ = poisson_glm_utils.convolve_with_basis(
                saccade_matrix,
                poisson_glm_config.SACCADE_BASIS,
                poisson_glm_config.SACCADE_DURATION_MS,
                poisson_glm_config.SACCADE_SPACING_MS,
                effect= poisson_glm_config.SACCADE_EFFECT
            )

            # Assign to choice-specific features
            choice_idx = int(trial['choice'])
            state_idx = int(trial['state'])
            choice_start = feature_idx['saccade_start'] + choice_idx * poisson_glm_config.SACCADE_N_BASES + state_idx * poisson_glm_config.SACCADE_N_BASES * poisson_glm_config.N_CHOICE_OPTIONS
            choice_end = choice_start + poisson_glm_config.SACCADE_N_BASES
            trial_design[:, choice_start:choice_end] = saccade_conv


        # 5. POST-SPIKE HISTORY COMPONENT
        history_matrix = poisson_glm_utils.create_post_spike_history_matrix(trial['spike_train'])
        trial_design[:, feature_idx['history_start']:feature_idx['history_end']] = history_matrix

        # 6. INTERCEPT TERM
        trial_design[:, feature_idx['intercept_idx']] = 1.0

        # Store processed trial
        trial_matrices.append(csr_matrix(trial_design))
        trial_spike_trains.append(trial['spike_train'])

    X = vstack(trial_matrices, format='csr')
    y = np.concatenate(trial_spike_trains)

    return X, y

def fit_poisson_glm(X, y):
    if issparse(X):
        X = X.toarray()#.astype(np.float16)  # Convert to dense for faster computation in this case
    # Strategy 2: Feature standardization for better conditioning
    X_means = X.mean(axis=0)
    X_stds = X.std(axis=0)

    # Avoid division by zero
    X_stds[X_stds < 1e-8] = 1.0

    # Standardize all features except intercept
    X_scaled = X.copy()
    X_scaled[:, :-1] = (X[:, :-1] - X_means[:-1]) / X_stds[:-1]

    def loss_fun(w):
        eta = X_scaled @ w
        if np.any(y[eta < -15]>0):
            return 1e20  # Penalty if rate is 0 but spikes are present
        else:
            eta = np.clip(eta, -15, 15)  # Prevent overflow
            mu = np.exp(eta)
            return np.sum(mu) - np.dot(y, eta) + 0.1 * np.dot(w, w)

    def grad_fun(w):
        eta = X_scaled @ w
        eta = np.clip(eta, -15, 15)
        mu = np.exp(eta)
        return X_scaled.T @ (mu - y) + 0.02 * w

    n_features = X_scaled.shape[1]
    w_init = np.zeros(n_features)
    w_init[-1] = np.log(max(y.mean(), 1e-8))  # Smart intercept

    # print(f"\nOptimizing {n_features} parameters...")
    start_time = time.time()

    result = minimize(
        fun=loss_fun,
        x0=w_init,
        method='L-BFGS-B',
        jac=grad_fun,
        options={
            'maxiter': 1000,    # Very limited iterations
            'gtol': 1e-3,      # Relaxed tolerance
            'ftol': 1e-5,      # Relaxed tolerance
            'maxfun': 200      # Limit function calls
        }
    )

    fit_time = time.time() - start_time
    # print(f"Optimization completed in {fit_time:.1f} seconds")

    if result.success or result.fun < 1e6:
        weights_scaled = result.x

        # don't save for memory
        # predicted y
        eta = X_scaled @ weights_scaled
        predicted = np.exp(np.clip(eta, -15, 15))
        result['predicted_y'] = predicted
        # result['X_scaled'] = X_scaled

        # result["mean_squared_error"] = np.mean((y - predicted)**2)
        # Transform weights back to original scale
        # fitted_weights = weights_scaled.copy()
        # fitted_weights[:-1] = weights_scaled[:-1] / X_stds[:-1]
        # fitted_weights[-1] = weights_scaled[-1] - np.dot(weights_scaled[:-1], X_means[:-1] / X_stds[:-1])
        # result["fitted_weights"] = fitted_weights
    else:
        print(f"Optimization failed: {result.message}")
        return None

    return result

def predict_poisson_glm(X, model):
    if issparse(X):
        X = X.toarray()#.astype(np.float16)
    X_means = X.mean(axis=0)
    X_stds = X.std(axis=0)

    # Avoid division by zero
    X_stds[X_stds < 1e-8] = 1.0

    X_scaled = X.copy()
    X_scaled[:, :-1] = (X[:, :-1] - X_means[:-1]) / X_stds[:-1]

    eta = X_scaled @ model.x
    return np.exp(np.clip(eta, -15, 15))


# 1. Train set visualization

In [ ]:
result_neuron_dict = {}

In [ ]:
#  read all file and filename in the result_dir
for file_path in result_dir.glob('*.pkl'):
    neuron_id = file_path.stem
    if neuron_id in result_neuron_dict:
        continue
    else:
        with open(file_path, 'rb') as f:
            fitting_result = pickle.load(f)
            result_neuron_dict[neuron_id] = fitting_result

In [ ]:
alignment_settings = {
    "visual": { "event": "target_onset", "start_time_ms": -50, "end_time_ms": 250 },
    "cue": { "event": "stimulus_onset", "start_time_ms": -100, "end_time_ms": 800 },
    "response": { "event": "response_onset", "start_time_ms": -300, "end_time_ms": 0 },
}
# coh_levels = result['coh_levels']

In [ ]:
def reconstruct_kernels_from_weights(weights, config):
    """
    Reconstruct temporal kernels from GLM weights using organized configuration.

    Args:
        weights: Fitted GLM weights (excluding intercept)
        config: GLMConfig instance with all model parameters

    Returns:
        kernels: Dictionary containing reconstructed temporal kernels
    """
    kernels = {}

    # Calculate feature indices using config
    target_start = 0
    target_end = config.FEATURES_TARGET

    stim_start = target_end
    stim_end = stim_start + config.FEATURES_STIMULUS

    stim_offset_start = stim_end
    stim_offset_end = stim_offset_start + config.FEATURE_STIMULUS_OFFSET

    saccade_start = stim_offset_end
    saccade_end = saccade_start + config.FEATURES_SACCADE

    history_start = saccade_end
    history_end = history_start + config.FEATURES_HISTORY


    # 1. TARGET ONSET KERNEL
    target_basis = poisson_glm_utils.make_smooth_temporal_basis(
        config.TARGET_DURATION_MS,
        filter_type=config.TARGET_BASIS,
        center_spacing=config.TARGET_SPACING_MS,
    )
    for state_idx in range(config.N_BIAS_STATES):
        state_target_start = target_start + state_idx * config.TARGET_N_BASES
        state_target_end = state_target_start + config.TARGET_N_BASES
        state_target_weights = weights[state_target_start:state_target_end]
        kernels[f'target_state_{state_idx}'] = {
            'kernel': target_basis @ state_target_weights,
            'time': np.arange(config.TARGET_DURATION_MS) if config.TARGET_EFFECT == config.EFFECT_CAUSAL else np.arange(config.TARGET_DURATION_MS) * -1,
            'weights': state_target_weights,
            'basis': target_basis,
            'duration_ms': config.TARGET_DURATION_MS,
            'n_bases': config.TARGET_N_BASES
        }

    # 2. STIMULUS KERNELS (ALL COHERENCE LEVELS)
    stim_basis = poisson_glm_utils.make_smooth_temporal_basis(
        config.STIMULUS_DURATION_MS,
        filter_type=config.STIMULUS_BASIS,
        center_spacing=config.STIMULUS_SPACING_MS,
    )

    # Individual coherence-state kernels
    for state_idx in range(config.N_BIAS_STATES):
        for coh_idx in range(config.N_COHERENCE_LEVELS):
            coh_state_start = stim_start + coh_idx * config.STIMULUS_N_BASES + state_idx * config.STIMULUS_N_BASES * config.N_COHERENCE_LEVELS
            coh_state_end = coh_state_start + config.STIMULUS_N_BASES
            stim_state_weights = weights[coh_state_start:coh_state_end]

            kernels[f'stimulus_coh_{coh_idx}_state_{state_idx}'] = {
                'kernel': stim_basis @ stim_state_weights,
                'time': np.arange(config.STIMULUS_DURATION_MS) if config.STIMULUS_EFFECT == config.EFFECT_CAUSAL else np.arange(config.STIMULUS_DURATION_MS) * -1,
                'weights': stim_state_weights,
                'basis': stim_basis,
                'duration_ms': config.STIMULUS_DURATION_MS,
                'n_bases': config.STIMULUS_N_BASES
            }

    # Average stimulus kernel
    all_stim_weights = weights[stim_start:stim_end].reshape(
        config.N_COHERENCE_LEVELS * config.N_BIAS_STATES,
        config.STIMULUS_N_BASES,
    )
    avg_stim_weights = np.mean(all_stim_weights, axis=0)
    kernels['stimulus'] = {
        'kernel': stim_basis @ avg_stim_weights,
        'time': np.arange(config.STIMULUS_DURATION_MS) if config.STIMULUS_EFFECT == config.EFFECT_CAUSAL else np.arange(config.STIMULUS_DURATION_MS) * -1,
        'weights': avg_stim_weights,
        'basis': stim_basis,
        'duration_ms': config.STIMULUS_DURATION_MS,
        'n_bases': config.STIMULUS_N_BASES
    }

    # 3. CHOICE/SACCADE KERNELS
    saccade_basis = poisson_glm_utils.make_smooth_temporal_basis(
        config.SACCADE_DURATION_MS,
        filter_type=config.SACCADE_BASIS,
        center_spacing=config.SACCADE_SPACING_MS,
    )
    for state_idx in range(config.N_BIAS_STATES):
        for choice_idx in range(config.N_CHOICE_OPTIONS):
            choice_state_start = saccade_start + choice_idx * config.SACCADE_N_BASES + state_idx * config.SACCADE_N_BASES * config.N_CHOICE_OPTIONS
            choice_state_end = choice_state_start + config.SACCADE_N_BASES
            choice_state_weights = weights[choice_state_start:choice_state_end]

            kernels[f'choice_{choice_idx}_state_{state_idx}'] = {
                'kernel': saccade_basis @ choice_state_weights,
                'time': np.arange(0, config.SACCADE_DURATION_MS) if config.SACCADE_EFFECT == config.EFFECT_CAUSAL else np.arange(0, config.SACCADE_DURATION_MS) * -1,
                'weights': choice_state_weights,
                'basis': saccade_basis,
                'duration_ms': config.SACCADE_DURATION_MS,
                'n_bases': config.SACCADE_N_BASES
            }

    # 4. POST-SPIKE HISTORY KERNEL
    history_weights = weights[history_start:history_end]
    history_basis = poisson_glm_utils.make_post_spike_history_basis(
        duration_uniform_ms=config.HISTORY_UNIFORM_MS,
        n_uniform_bases=config.HISTORY_N_UNIFORM,
        duration_nonlinear_ms=config.HISTORY_NONLINEAR_MS,
        n_nonlinear_bases=config.HISTORY_N_NONLINEAR
    )

    kernels['history'] = {
        'kernel': history_basis @ history_weights,
        'time': np.arange(config.HISTORY_UNIFORM_MS + config.HISTORY_NONLINEAR_MS),
        'weights': history_weights,
        'basis': history_basis,
        'uniform_duration_ms': config.HISTORY_UNIFORM_MS,
        'nonlinear_duration_ms': config.HISTORY_NONLINEAR_MS,
        'n_uniform_bases': config.HISTORY_N_UNIFORM,
        'n_nonlinear_bases': config.HISTORY_N_NONLINEAR,
        'total_duration_ms': config.HISTORY_UNIFORM_MS + config.HISTORY_NONLINEAR_MS
    }

    # SUMMARY INFORMATION
    kernels['_config_summary'] = {
        'total_features_used': len(weights),
        'expected_features': config.get_total_features() - 1,  # Excluding intercept
        'feature_breakdown': {
            'target': config.FEATURES_TARGET,
            'stimulus': config.FEATURES_STIMULUS,
            'saccade': config.FEATURES_SACCADE,
            'history': config.FEATURES_HISTORY
        },
        'temporal_parameters': {
            'target_duration_ms': config.TARGET_DURATION_MS,
            'stimulus_duration_ms': config.STIMULUS_DURATION_MS,
            'saccade_duration_ms': config.SACCADE_DURATION_MS,
            'history_duration_ms': config.HISTORY_UNIFORM_MS + config.HISTORY_NONLINEAR_MS
        },
        'basis_parameters': {
            'target_n_bases': config.TARGET_N_BASES,
            'stimulus_n_bases': config.STIMULUS_N_BASES,
            'saccade_n_bases': config.SACCADE_N_BASES,
            'history_n_bases': config.HISTORY_N_UNIFORM + config.HISTORY_N_NONLINEAR
        }
    }

    return kernels

In [ ]:
def match_ylim(ax_list):
    """Unify y-axis limits across multiple subplots based on their current data range."""
    ymins = [ax.get_ylim()[0] for ax in ax_list]
    ymaxs = [ax.get_ylim()[1] for ax in ax_list]
    common_ylim = (min(ymins), max(ymaxs))
    for ax in ax_list:
        ax.set_ylim(common_ylim)


def plot_kernels(neuron_id, result, coh_levels):
    fitted_weights = result['x'][:-1]  # Exclude intercept
    kernels = reconstruct_kernels_from_weights(fitted_weights, poisson_glm_config)
    # alpha = {0: 0.5, 1: 1.0}
    alpha = {0: 1.0, 1: 1.0}

    fig = plt.figure(figsize=(20, 10*poisson_glm_config.N_BIAS_STATES))
    gs = fig.add_gridspec(4, 3, height_ratios=[1, 1, 1, 1], width_ratios=[1, 1, 1.2])

    # ========================================================================
    # 1. TARGET ONSET KERNEL
    # ========================================================================
    target_axs = []
    for state in range(poisson_glm_config.N_BIAS_STATES):
        target_axs.append(fig.add_subplot(gs[state, 0]))
    for ax, state_idx in zip(target_axs, range(poisson_glm_config.N_BIAS_STATES)):
        state_kernel = kernels[f'target_state_{state_idx}']
        ax.plot(state_kernel['time'], state_kernel['kernel'], linewidth=2, color='k',
                alpha=alpha[state_idx], label=f'State {state_idx}')
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_title(f'Target Component ({state_kernel["n_bases"]} bases, {state_kernel["duration_ms"]}ms)',
                      fontsize=11, fontweight='bold')
        ax.legend(fontsize=8, loc='upper right', ncol=2)
        ax.set_ylabel('Filter amplitude')
        ax.set_xlabel('Time (ms)')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, state_kernel["duration_ms"])
    match_ylim(target_axs)  # ✅ common y-limits

    # ========================================================================
    # 2. STIMULUS KERNELS
    # ========================================================================
    stimulus_axs = []
    for state in range(poisson_glm_config.N_BIAS_STATES):
        stimulus_axs.append(fig.add_subplot(gs[state, 1]))
    colors_coh = ["#AC3626","#EC6A50","#EF8D41", "#96D6EC", "#6FC3EB", "#5289C6", "#4469B1"]
    for ax, state_idx in zip(stimulus_axs, range(poisson_glm_config.N_BIAS_STATES)):
        for coh_idx in range(poisson_glm_config.N_COHERENCE_LEVELS):
            kernel_key = f'stimulus_coh_{coh_idx}_state_{state_idx}'
            if kernel_key in kernels:
                coh_kernel = kernels[kernel_key]
                ax.plot(coh_kernel['time'], coh_kernel['kernel'],
                        color=colors_coh[coh_idx], linewidth=2, alpha=alpha[state_idx],
                        label=f'{coh_levels[coh_idx]*100}%')
        stim_avg = kernels['stimulus']
        ax.plot(stim_avg['time'], stim_avg['kernel'], 'k--', linewidth=3, alpha=0.9, label='Average')
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_title(f'Stimulus Component ({stim_avg["n_bases"]} bases × {poisson_glm_config.N_COHERENCE_LEVELS} coh, {stim_avg["duration_ms"]}ms)',
                      fontsize=11, fontweight='bold')
        ax.set_ylabel('Filter amplitude')
        ax.set_xlabel('Time (ms)')
        ax.legend(fontsize=8, loc='upper right', ncol=2)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, stim_avg["duration_ms"])
    match_ylim(stimulus_axs)

    # ========================================================================
    # 3. CHOICE/SACCADE KERNELS
    # ========================================================================
    choice_axs = []
    for state in range(poisson_glm_config.N_BIAS_STATES):
        choice_axs.append(fig.add_subplot(gs[state, 2]))

    choice_colors = ['dodgerblue', 'crimson']
    choice_labels = ['Choice 0', 'Choice 1']

    for ax, state_idx in zip(choice_axs, range(poisson_glm_config.N_BIAS_STATES)):
        for choice_idx in range(poisson_glm_config.N_CHOICE_OPTIONS):
            choice_kernel = kernels[f'choice_{choice_idx}_state_{state_idx}']
            ax.plot(choice_kernel['time'], choice_kernel['kernel'],
                    color=choice_colors[choice_idx], linewidth=3, alpha=alpha[state_idx],
                    label=choice_labels[choice_idx])
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.axvline(0, color='gray', linestyle=':', alpha=0.7, label='Saccade')
        ax.set_title(f'Choice Component ({choice_kernel["n_bases"]} bases × {poisson_glm_config.N_CHOICE_OPTIONS} choices, {choice_kernel["duration_ms"]}ms)',
                      fontsize=11, fontweight='bold')
        ax.set_ylabel('Filter amplitude')
        ax.set_xlabel('Time relative to saccade (ms)')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(choice_kernel["time"].min(), 0)
    match_ylim(choice_axs)

    # ========================================================================
    # 4. POST-SPIKE HISTORY KERNEL
    # ========================================================================
    ax5 = fig.add_subplot(gs[poisson_glm_config.N_BIAS_STATES, 1])
    history_kernel = kernels['history']
    ax5.plot(history_kernel['time'], history_kernel['kernel'], 'purple', linewidth=3, alpha=0.8)
    ax5.axhline(0, color='gray', linestyle='--', alpha=0.5)
    # Mark uniform vs nonlinear regions
    ax5.axvline(history_kernel['uniform_duration_ms'], color='red', linestyle=':', alpha=0.7,
               label=f'Uniform→Nonlinear ({history_kernel["uniform_duration_ms"]}ms)')
    ax5.set_title(f'Post-Spike History ({history_kernel["n_uniform_bases"]}+{history_kernel["n_nonlinear_bases"]} bases, {history_kernel["total_duration_ms"]}ms)',
                  fontsize=11, fontweight='bold')
    ax5.set_ylabel('Filter amplitude')
    ax5.set_xlabel('Time after spike (ms)')
    ax5.legend(fontsize=9)
    ax5.grid(True, alpha=0.3)
    ax5.set_xlim(0, history_kernel["total_duration_ms"])



    plt.suptitle(f"Poisson GLM fitting for neuron {neuron_id}")
    plt.tight_layout()
    plt.subplots_adjust(top=0.94)
    plt.show()


In [ ]:
for neuron_id, fitting_result in result_neuron_dict.items():
    if int(neuron_id) == 141:
        for fold_result in fitting_result["folds"]:
            plot_kernels(neuron_id, fold_result["model"], coh_levels)
    # break

# 2. PSTH prediction

In [ ]:
def spike_train_convolved(trial_info, pred, alignment_settings, alignment_buffer=0):
    convolved_spike_density = []
    predicted_spike_rate = []
    sigma = 10  # 10ms smoothing

    for alignment_idx, key in enumerate(alignment_settings):
        convolved_spike_density.append(np.zeros([len(trial_info), alignment_settings[key]["end_time_ms"] - alignment_settings[key]["start_time_ms"] + 2 * alignment_buffer]))
        predicted_spike_rate.append(np.zeros([len(trial_info), alignment_settings[key]["end_time_ms"] - alignment_settings[key]["start_time_ms"]]))

        for idx in range(len(trial_info)):
            start_timestamp = int(trial_info.loc[idx, alignment_settings[key]["event"]] + (alignment_settings[key]["start_time_ms"] - alignment_buffer))
            end_timestamp = int(trial_info.loc[idx, alignment_settings[key]["event"]] + (alignment_settings[key]["end_time_ms"] + alignment_buffer))

            trial_spike_train = trial_info.loc[idx, "spike_train"]
            trial_predicted_rate = pred[idx]

            end_timestamp = min(end_timestamp, len(trial_spike_train))
            trial_spike_train = trial_spike_train[start_timestamp:end_timestamp]
            trial_predicted_rate = trial_predicted_rate[start_timestamp + alignment_buffer:end_timestamp-alignment_buffer] # not buffering for predicted rate
            convolved_spike_density[alignment_idx][idx, :len(trial_spike_train)] = gaussian_filter1d(trial_spike_train, sigma=sigma, truncate=3)  # gaussian smoothened
            predicted_spike_rate[alignment_idx][idx,:len(trial_predicted_rate)] = gaussian_filter1d(trial_predicted_rate, sigma=sigma, truncate=3)

            if alignment_settings[key]["event"] == "stimulus_onset":
                if (end_timestamp - alignment_buffer) > trial_info.loc[idx, "response_onset"] - 50 :  # exclude spikes after -50ms aligned to saccade
                    pre_saccade_idx = np.ceil((end_timestamp - trial_info.loc[idx, "response_onset"])+ 50 - alignment_buffer).astype(int)
                    convolved_spike_density[alignment_idx][idx, -pre_saccade_idx:] = np.nan
                    predicted_spike_rate[alignment_idx][-pre_saccade_idx-alignment_buffer:] = np.nan

            # elif alignment_settings[alignment_idx]['alignment_event'] == 'response_onset':
            #     if start_timestamp <

        convolved_spike_density[alignment_idx] = convolved_spike_density[alignment_idx][:, alignment_buffer:convolved_spike_density[alignment_idx].shape[1]-alignment_buffer] * 1000
        predicted_spike_rate[alignment_idx] *= 1000

    return convolved_spike_density, predicted_spike_rate


In [ ]:
def plot_data_and_model_psth(trial_info, pred, alignment_settings):
    convolved_spike_density, predicted_spike_rate = spike_train_convolved(trial_info, pred, alignment_settings)

    colors = ["#e31a1c", "#ff7f00", "#33a02c", "#1f78b4"]
    fig, axs = plt.subplots(2*poisson_glm_config.N_BIAS_STATES, len(alignment_settings), figsize=(25, 10*poisson_glm_config.N_BIAS_STATES))#, sharey='col')
    abs_coh_levels = np.sort(np.unique(np.abs(coh_levels)))

    for alignment_idx, key in enumerate(alignment_settings):
        for state in range(poisson_glm_config.N_BIAS_STATES):
            
            for choice in [0,1]:
                ax = axs[state*2+choice,alignment_idx]
            # res_ax = axs[state*2+1,alignment_idx]
                for coh_idx, coherence in enumerate(abs_coh_levels):
                    trial_number = trial_info[((trial_info.choice == choice) & (np.abs(trial_info.coherence) == coherence)) & (trial_info.state == state)].index
                    # trial_number = trial_info[((trial_info.choice == choice) & (np.abs(trial_info.coherence) == coherence))].index

                    if alignment_settings[key]["event"] == "stimulus_onset":
                        reaction_time = np.nanmedian(trial_info.loc[trial_number, "reaction_time"])
                        end = np.min([reaction_time - 50, alignment_settings[key]["end_time_ms"]]).astype(int) - alignment_settings[key]["start_time_ms"] + 1

                    else:
                        end = alignment_settings[key]["end_time_ms"] - alignment_settings[key]["start_time_ms"] + 1

                    x_plot = range(alignment_settings[key]["start_time_ms"], alignment_settings[key]["end_time_ms"] )


                    y_plot = np.nanmean(convolved_spike_density[alignment_idx][trial_number, :], axis=0)
                    y_plot[end:] = np.nan

                    y_pred_plot = np.nanmean(predicted_spike_rate[alignment_idx][trial_number, :], axis=0)
                    y_pred_plot[end:] = np.nan

                    ax.plot(x_plot, y_plot, color=colors[coh_idx], linestyle='-', linewidth=2)
                    ax.plot(x_plot, y_pred_plot, color=colors[coh_idx], linestyle='--', linewidth=2)
                    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
                    ax.grid(False)
                    # no right and top spines
                    ax.spines['right'].set_visible(False)
                    ax.spines['top'].set_visible(False)

                    # res_ax.plot(x_plot, y_plot - y_pred_plot, color=colors[coh_idx], linestyle='-', linewidth=2)
                    # res_ax.grid(False)
                    # res_ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
                    # # no right and top spines
                    # res_ax.spines['right'].set_visible(False)
                    # res_ax.spines['top'].set_visible(False)

        axs[0,alignment_idx].set_title(f"Aligned to {key} ({alignment_settings[key]['event'].replace('_',' ')})", fontsize=12, fontweight='bold')
    for coh_idx, coherence in enumerate(abs_coh_levels):
        axs[0,0].plot([], [], color=colors[coh_idx], linestyle='-', linewidth=2, label=f"{int(coherence*100)}% coh")
    # for state in range(poisson_glm_config.N_BIAS_STATES):)#
    # axs[2,0].set_ylabel(f"State 1 PSTH (Biased)", fontsize=12)
    # axs[3,0].set_ylabel(f"Residual (Biased)", fontsize=12)
    fig.legend(bbox_to_anchor=(0.4, 0.3, 0.5, 0.5), fontsize=12)


Train set

In [ ]:
for neuron_id, fitting_result in result_neuron_dict.items():
    if int(neuron_id) == 141:
        print(f"Neuron {neuron_id} PSTH")
        for fold_result in fitting_result["folds"]:
            trial_info = fold_result['train_data']
            pred = fold_result['train_predictions']
            plot_data_and_model_psth(trial_info, pred, alignment_settings)
    # break

Test set

In [ ]:
# trial duration of train vs test sets in bad neurons
for neuron_id, fitting_result in result_neuron_dict.items():
    if np.isin(neuron_id, bad_neurons):
        print(f"Neuron {neuron_id} PSTH")
        for fold_result in fitting_result["folds"]:
            print(f"Fold {fold_result['fold']}")
            test_trial_info = fold_result['test_data']
            test_pred = fold_result['test_predictions']
            train_trial_info = fold_result['train_data']
            fig,axs = plt.subplots(1,len(abs_coh_levels), figsize=(20,5))
            for idx, coh in enumerate(abs_coh_levels):
                print(f'Coherence {coh}:', 
                      f'{len(train_trial_info[abs(train_trial_info.coherence) == coh])} trials in train,',
                      f'{len(test_trial_info[abs(test_trial_info.coherence) == coh])} trials in test')
                # axs[idx].hist(train_trial_info[abs(train_trial_info.coherence) == coh].duration)
                # axs[idx].hist(test_trial_info[abs(test_trial_info.coherence) == coh].duration)
                axs[idx].hist(train_trial_info[abs(train_trial_info.coherence) == coh].response_onset - train_trial_info[abs(train_trial_info.coherence) == coh].stimulus_onset, alpha=0.5, label='train')
                axs[idx].hist(test_trial_info[abs(test_trial_info.coherence) == coh].response_onset - test_trial_info[abs(test_trial_info.coherence) == coh].stimulus_onset, alpha=0.5, label='test')
                axs[idx].set_title(f'Coherence {coh}')
            
        break

#### set one set of kernel weight to zero and see the predicted PSTH?
1. spike history
2. stimulus onset
3. stimulus offset
4. saccade
5. visual

In [ ]:
import copy

for neuron_id, fitting_result in result_neuron_dict.items():
    if int(neuron_id) == 31:
        print(f"Neuron {neuron_id} PSTH")
        for fold_result in fitting_result["folds"]:
            test_trial_info = fold_result['test_data']
            X_test, y_test = build_design_matrix(test_trial_info, coh_levels)
            # X_test[:,feature_idx['history_start']:feature_idx['history_end']] = 0  # zero out history terms
            # X_test[:,feature_idx['stim_start']:feature_idx['stim_end']] = 0  # zero out stim onset terms
            model = copy.deepcopy(fold_result["model"])
            
            model.x[feature_idx['target_start']:feature_idx['target_end']] = 0 # set specific term weights to zero
            flat_pred = predict_poisson_glm(X_test,model)
            pred, onset = [], 0
            for _, t in test_trial_info.iterrows():
                n = t["n_bins"]
                pred.append(flat_pred[onset:onset+n])
                onset += n
            plot_data_and_model_psth(test_trial_info, pred, alignment_settings) 

In [ ]:
for neuron_id, fitting_result in result_neuron_dict.items():
    if int(neuron_id) == 141:
        print(f"Neuron {neuron_id} PSTH")
        for fold_result in fitting_result["folds"]:
            trial_info = fold_result['test_data']
            pred = fold_result['test_predictions']
            plot_data_and_model_psth(trial_info, pred, alignment_settings)
        # break

# 3. Goodness of fit / Variance explained (for all trial PSTH?)

In [ ]:
def variance_explained(y_true, y_pred):
    """
    Compute variance explained for neural predictions.
    
    Parameters
    ----------
    y_true : array-like
        True spike counts or firing rate (can be per bin or per trial)
    y_pred : array-like
        Predicted spike counts or firing rate (same shape as y_true)
        
    Returns
    -------
    ve : float
        Variance explained, between -inf and 1.0
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Remove NaNs if any
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    # Compute residual variance
    residual_var = np.var(y_true - y_pred, ddof=0)
    total_var = np.var(y_true, ddof=0)

    ve = 1 - residual_var / total_var
    return ve

In [ ]:
# Goodness of fit/variance explained of stitched PSTH over each coherence and choice pair
# stich PSTH over 5 test sets
# from scipy.stats import pearsonr, spearmanr
# from sklearn.metrics import r2_score


abs_coh_levels = np.sort(np.unique(np.abs(coh_levels)))
r2_fr_list = {key:{'r2':[], 'mean_fr':[]} for key in alignment_settings}
subjects = []

for neuron_id, fitting_result in result_neuron_dict.items():
    trial_info = pd.DataFrame()
    test_pred = []
    for fold_result in fitting_result["folds"]:
        test_pred.extend(fold_result['test_predictions'])
        trial_info = pd.concat([trial_info, fold_result['test_data']], ignore_index=True)

    # stitch test predictions across all five folds
    convolved_spike_density, predicted_spike_rate = spike_train_convolved(trial_info, test_pred, alignment_settings)

    for alignment_idx, key in enumerate(alignment_settings):
        # for state in range(poisson_glm_config.N_BIAS_STATES):
            
        #     for choice in [0,1]:
        #         for coh_idx, coherence in enumerate(abs_coh_levels):
        #             trial_number = trial_info[((trial_info.choice == choice) & (np.abs(trial_info.coherence) == coherence)) & (trial_info.state == state)].index
                    # trial_number = trial_info[((trial_info.choice == choice) & (np.abs(trial_info.coherence) == coherence))].index

                    # if alignment_settings[key]["event"] == "stimulus_onset":
                    #     reaction_time = np.nanmedian(trial_info.loc[trial_number, "reaction_time"])
                    #     end = np.min([reaction_time - 50, alignment_settings[key]["end_time_ms"]]).astype(int) - alignment_settings[key]["start_time_ms"] + 1

                    # else:
                    #     end = alignment_settings[key]["end_time_ms"] - alignment_settings[key]["start_time_ms"] + 1

                    
                    # y_plot = np.nanmean(convolved_spike_density[alignment_idx][trial_number, :], axis=0)
                    # # y_plot[end:] = np.nan

                    # y_pred_plot = np.nanmean(predicted_spike_rate[alignment_idx][trial_number, :], axis=0)
                    # y_pred_plot[end:] = np.nan

        # compute R^2 between convolved_spike_density and predicted_spike_rate (overall PSTH)
        # r2 = r2_score(np.nanmean(convolved_spike_density[alignment_idx], axis=0), np.nanmean(predicted_spike_rate[alignment_idx], axis=0))
        r2 = variance_explained(np.nanmean(convolved_spike_density[alignment_idx], axis=0), np.nanmean(predicted_spike_rate[alignment_idx], axis=0))    
        # print(f"Variance explained (R^2) for neuron {neuron_id} aligned to {key}: {r2:.4f}")
        r2_fr_list[key]['r2'].append(r2)
        r2_fr_list[key]['mean_fr'].append(np.nanmean(convolved_spike_density[alignment_idx]))

In [ ]:
fig,axs = plt.subplots(1, len(alignment_settings), figsize=(20,5))
for idx, key in enumerate(r2_fr_list):
    axs[idx].scatter(r2_fr_list[key]['mean_fr'], r2_fr_list[key]['r2'], alpha=0.5, s=15, color='k')
    axs[idx].set_xlabel('Mean Firing Rate (spikes/s)')
    axs[idx].set_ylabel('Variance Explained (R^2)')
    axs[idx].set_ylim(-0.02,1.02)
    axs[idx].set_title(f"{key} Epoch")
    axs[idx].grid(True, alpha=0.5)

In [ ]:

r2_fr_list_stitched = {'r2':[], 'mean_fr':[]}

for neuron_id, fitting_result in result_neuron_dict.items():
    trial_info = pd.DataFrame()
    test_pred = []
    for fold_result in fitting_result["folds"]:
        test_pred.extend(fold_result['test_predictions'])
        trial_info = pd.concat([trial_info, fold_result['test_data']], ignore_index=True)

    # stitch test predictions across all five folds
    convolved_spike_density, predicted_spike_rate = spike_train_convolved(trial_info, test_pred, alignment_settings)

    
    # stich all three alignments together

        # compute R^2 between convolved_spike_density and predicted_spike_rate (overall PSTH)
        # r2 = r2_score(np.nanmean(convolved_spike_density[alignment_idx], axis=0), np.nanmean(predicted_spike_rate[alignment_idx], axis=0))
    r2 = variance_explained(np.nanmean(np.hstack(convolved_spike_density), axis=0), np.nanmean(np.hstack(predicted_spike_rate), axis=0))    
    # print(f"Variance explained (R^2) for neuron {neuron_id} aligned to {key}: {r2:.4f}")
    r2_fr_list_stitched['r2'].append(r2)
    r2_fr_list_stitched['mean_fr'].append(np.nanmean(np.hstack(convolved_spike_density)))

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4,5))
ax.scatter(r2_fr_list_stitched['mean_fr'], r2_fr_list_stitched['r2'], alpha=0.5, s=15, color='k')
ax.set_xlabel('Mean Firing Rate (spikes/s)')
ax.set_ylabel('Variance Explained (R^2)')
ax.set_ylim(-2,1.02)
ax.axhline(0.8, color='b', linestyle='--', alpha=0.5)
# ax.set_title(f"{key} Epoch")
ax.grid(True, alpha=0.5)

In [ ]:
neuron_ids = list(result_neuron_dict.keys())
bad_neurons = []
for idx, r2 in enumerate(r2_fr_list_stitched['r2']):
    if r2 < 0:
        print(f"Neuron {neuron_ids[idx]} has negative R^2: {r2:.4f}")
        bad_neurons.append(neuron_ids[idx])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4, 5))
type_colors = {'visual_motor': 'red',
               'motor': 'blue', 
               'visual_tonic': 'green', 
               'visual_phasic': 'purple',
                'unknown': 'gray', 
                'buildup': 'orange'}
plotted_types = set()

neuron_ids = list(result_neuron_dict.keys())

for i, neuron_id in enumerate(neuron_ids):
    if r2_fr_list_stitched['mean_fr'][i] > 0:
        neuron_type = neuron_metadata.loc[int(neuron_id), 'classification']
        color = type_colors.get(neuron_type, 'k')

        label = neuron_type if neuron_type not in plotted_types else None
        plotted_types.add(neuron_type)

        ax.scatter(
            r2_fr_list_stitched['mean_fr'][i],
            r2_fr_list_stitched['r2'][i],
            color=color,
            s=15,
            alpha=0.6,
            label=label
        )
ax.set_xlabel('Mean Firing Rate (spikes/s)')
ax.set_ylabel('Variance Explained (R^2)')
ax.set_ylim(-20,1.02)
ax.axhline(0, color='b', linestyle='--', alpha=0.5)
# ax.set_title(f"{key} Epoch")
ax.grid(True, alpha=0.5)
ax.legend()

In [ ]:

# logL(λ;r)=t∑​[r(t)logΔλ(t)−Δλ(t)]
def ll_spike_train(spike_rate, predicted_rate, n_trials, ndt=0.001):
    # spike_rate and predicted_rate are both 1D arrays of n (10ms)time bins across all trials?
    eps = 1e-10
    log_likelihood = np.sum(spike_rate * np.log(predicted_rate*ndt + eps) - predicted_rate*ndt)
    return log_likelihood / n_trials

# then average ll over 5 test sets

In [ ]:
# Spike prediction accuracy: 
# taking the difference between the full model log-likelihood and the log-likelihood of a (single parameter) homogeneous Poisson model normalized by the number of spikes on the  cross-validation set


## Verification

In [ ]:
# def spike_train_convolved_array(trial_info):
#     convolved_spike_density = []
#     predicted_spike_rate = []
#     sigma = 10  # 10ms smoothing

#     for idx in range(len(trial_info)):
#         trial_spike_train = trial_info.loc[idx, "spike_train"]
#         trial_predicted_rate = trial_info.loc[idx, "predicted_spike_rate"]

#         convolved_spike_density.append(gaussian_filter1d(trial_spike_train, sigma=sigma, truncate=3) * 1000) # gaussian smoothened
#         predicted_spike_rate.append(gaussian_filter1d(trial_predicted_rate, sigma=sigma, truncate=3) * 1000)

#     # convert to numpy array and make one long array
#     convolved_spike_density = np.concatenate(convolved_spike_density)
#     predicted_spike_rate = np.concatenate(predicted_spike_rate)
#     return convolved_spike_density, predicted_spike_rate


In [ ]:
# from scipy.stats import pearsonr, spearmanr
# from sklearn.metrics import r2_score

# for neuron_id, result in fitting_result.items():
#     trial_info = result['trial_info']
#     spike_train, predicted_spike_train = spike_train_convolved_array(trial_info)

#     # Remove any NaN values for correlation calculations
#     valid_indices = ~(np.isnan(spike_train) | np.isnan(predicted_spike_train))
#     spike_train_clean = spike_train[valid_indices]
#     predicted_spike_train_clean = predicted_spike_train[valid_indices]

#     # 1. Pearson correlation (linear relationship)
#     pearson_r, pearson_p = pearsonr(spike_train_clean, predicted_spike_train_clean)

#     # 2. Spearman correlation (monotonic relationship)
#     spearman_r, spearman_p = spearmanr(spike_train_clean, predicted_spike_train_clean)

#     # 3. R-squared (coefficient of determination - variance explained)
#     r2 = r2_score(spike_train_clean, predicted_spike_train_clean)

#     # 4. Mean Squared Error
#     mse = np.mean((spike_train_clean - predicted_spike_train_clean) ** 2)

#     # 5. Root Mean Squared Error
#     rmse = np.sqrt(mse)

#     # 6. Mean Absolute Error
#     mae = np.mean(np.abs(spike_train_clean - predicted_spike_train_clean))

#     # 7. Explained variance (alternative calculation)
#     explained_variance = 1 - (np.var(spike_train_clean - predicted_spike_train_clean) / np.var(spike_train_clean))

#     print(f"\n{'='*60}")
#     print(f"MODEL FIT STATISTICS FOR NEURON {neuron_id}")
#     print(f"{'='*60}")
#     print(f"Data points used: {len(spike_train_clean):,}")
#     print(f"")
#     print(f"CORRELATION MEASURES:")
#     print(f"  Pearson correlation:  r = {pearson_r:.4f}, p = {pearson_p:.2e}")
#     print(f"  Spearman correlation: ρ = {spearman_r:.4f}, p = {spearman_p:.2e}")
#     print(f"")
#     print(f"VARIANCE EXPLAINED:")
#     print(f"  R² (coefficient of determination): {r2:.4f} ({r2*100:.1f}% variance explained)")
#     print(f"  Explained variance (alternative):   {explained_variance:.4f} ({explained_variance*100:.1f}%)")
#     print(f"")
#     print(f"ERROR METRICS:")
#     print(f"  Mean Squared Error (MSE):  {mse:.4f}")
#     print(f"  Root Mean Squared Error:   {rmse:.4f}")
#     print(f"  Mean Absolute Error:       {mae:.4f}")
#     print(f"")
#     print(f"DATA SUMMARY:")
#     print(f"  Actual spike rate - Mean: {np.mean(spike_train_clean):.2f}, Std: {np.std(spike_train_clean):.2f}")
#     print(f"  Predicted rate    - Mean: {np.mean(predicted_spike_train_clean):.2f}, Std: {np.std(predicted_spike_train_clean):.2f}")

#     # Create scatter plot for visualization
#     plt.figure(figsize=(12, 5))

#     # Subplot 1: Scatter plot
#     plt.subplot(1, 2, 1)
#     plt.scatter(spike_train_clean, predicted_spike_train_clean, alpha=0.6, s=1)
#     plt.plot([0, max(spike_train_clean.max(), predicted_spike_train_clean.max())],
#              [0, max(spike_train_clean.max(), predicted_spike_train_clean.max())],
#              'r--', alpha=0.8, label='Perfect fit')
#     plt.xlabel('Actual Spike Rate (Hz)')
#     plt.ylabel('Predicted Spike Rate (Hz)')
#     plt.title(f'Neuron {neuron_id}: Actual vs Predicted\nr = {pearson_r:.3f}, R² = {r2:.3f}')
#     plt.legend()
#     plt.grid(True, alpha=0.3)

#     # Subplot 2: Residuals plot
#     plt.subplot(1, 2, 2)
#     residuals = spike_train_clean - predicted_spike_train_clean
#     plt.scatter(predicted_spike_train_clean, residuals, alpha=0.6, s=1)
#     plt.axhline(y=0, color='r', linestyle='--', alpha=0.8)
#     plt.xlabel('Predicted Spike Rate (Hz)')
#     plt.ylabel('Residuals (Actual - Predicted)')
#     plt.title(f'Residuals vs Predicted\nRMSE = {rmse:.3f}')
#     plt.grid(True, alpha=0.3)

#     plt.tight_layout()
#     plt.show()